# 4. Physical stage from saved warmup
Load notebook 3's immutable warmup and exact inputs. Disable warmup and start physical factorization from those fields. Each outer round alternates ER updates with a shared charge/magnetization projection; set OUTER_ITERATIONS=0 for a **single final physical projection without detector updates**.

Saturation anchors are optional: list scan indices independently known to be saturated and their signs. Without two opposite saturated anchors the fitted magnetization is relative contrast, not an absolute calibrated magnetization. The material region defaults to the largest support component saved in notebook 3. Illumination and phase switches are independent.
Saturated frames are **not required**. For these opposite-helicity data, keep both saturation lists empty. The +1/-1 polarization coefficients describe helicity, not saturated magnetization. The two helicities of each pair share one unknown state magnetization. Fitted magnetization and its trend are relative unless an independent magnetic response or saturation calibration fixes their scale.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow

from library import phase_retrieval_universal as universal

In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
INPUT = PROCESSED / "03_warmup.h5"  # HDF5 input from the preceding notebook.
OUTPUT = PROCESSED / "04_physical.h5"  # HDF5 file written by this notebook; overwritten on rerun.

# Prepared hologram grid (must match the saved warmup)
CROP = 0  # Pixels removed from each edge; keep 0 in retrieval to preserve the saved grid.
BINNING = 1  # Detector binning factor; keep 1 for the prepared grid.
ROI = None  # Optional retrieval region; keep None to use the prepared grid.
HOLOGRAM_INTENSITY_CUTOFF_VMIN = -1  # -1 disables percentile background subtraction.

# Initialization: fields and modes are loaded from notebook 3

# Modes and coherence
PARTIAL_COHERENCE = False  # Enable partial-coherence updates; False uses coherent retrieval.
FFT_WORKERS = 1  # FFT threads used for each observation update.
COMMON_SECONDARY_MODES = True  # Share higher-mode objects across observations.

# Warmup: disabled because starting fields are loaded
WARMUP_NIT = 0  # 0 skips warmup and starts from the saved fields.

# Physical model and alternating steps
PROJECTION_MODEL = "physical_factorized"  # Physical factorization model; "none" disables physical fitting.
OUTER_ITERATIONS = 10        # 0: projection only; >0: ER + physical projection rounds.
INNER_MODE = "ER"  # Detector/support update algorithm between physical projections.
INNER_ITERATIONS = 50  # Retrieval iterations per observation in each outer round.
BETA_MODE = "const"  # Schedule controlling the feedback parameter beta.
BETA_ZERO = .5  # Initial feedback parameter beta.
ALPHA_ZERO = .5  # Initial TV regularization step size.
TV_FREQ = 10  # Apply TV regularization every this many retrieval iterations.
AVERAGE_IMG = 1  # Number of final iteration fields averaged per update.

PHYSICAL_ITERATIONS = 5  # Alternating charge/magnetization fit iterations per projection.
PROJECTION_RELAXATION = 1.  # Physical projection mixing; 1 fully applies the fitted model.
FINAL_PROJECTION_RELAXATION = 1.  # Mixing for the final physical projection; 0 skips it.
FINAL_FOURIER_CONSTRAINT = False  # Reapply measured amplitudes after the physical fit if True.

# Material and magnetization constraints
SATURATED_SCAN_INDICES = []  # Leave empty: these data have no saturated states.
SATURATION_SIGNS = []  # Leave empty; helicity signs belong to POLARIZATIONS, not here.
FIT_MATERIAL_THICKNESS = False  # Fit thickness if True; otherwise retain the saved material map.
PHYSICAL_PROJECTION_OBJECT_ROI = True  # Fit on the material region and shift-reference pixels.
PHYSICAL_PHASE_REFERENCE = True  # Choose consistent logarithm phase branches.
INITIAL_BINARY_MAGNETIZATION = False  # Continuous measured-contrast initializer by default.
BINARY_MAGNETIZATION = False  # Constrain every fitted magnetization update to discrete values.
CLIP_MAGNETIZATION = False  # Bound fitted magnetization to [-1, 1] if True.
MAGNETIZATION_TREND = "none"  # none, increasing, decreasing; observation order.
MAGNETIZATION_TREND_INTENSITY_CUTOFF = 0.01  # Ignore pixels below 1% of peak pooled exit-wave intensity.
MAGNETIZATION_TREND_CORRECTION = "offset"  # "offset", "factor", or "affine_domains"; infeasible corrections use offset.
BOUND_MAGNETIZATION_MEDIAN = False  # True bounds each weighted state median to [-1, 1], even with no trend.


# Physical projection focus (None inherits the choice saved in notebook 03)
FOCUS_PROP_UM = None  # Propagation in micrometres; 0 disables propagation.
FOCUS_PHASE_RAD = None  # Phase rotation in radians; 0 disables rotation.
DETECTOR_PIXEL_SIZE_M = None  # Detector pixel size in metres; None uses saved geometry.
DETECTOR_DISTANCE_M = None  # Sample-to-detector distance in metres; None uses saved geometry.

# Illumination fits and spatial weighting
ALIGN_GLOBAL_PHASE = True  # Align observation phase gauges before physical fitting.
FIT_ILLUMINATION_INTENSITY = True  # Fit relative illumination using material-free support pixels.
FIT_APERTURE_ILLUMINATION = True  # Fit separate illumination factors for reference holes.
OBJECT_ILLUMINATION_MODE = "linear_plane"  # RH-based gain extrapolation to object.
APERTURE_FACTOR_TYPE = "complex"  # "complex" fits amplitude and phase; "amplitude" fits amplitude only.


In [ ]:
with h5py.File(INPUT, "r") as h:
    warmup = h["warmup_fields"][()]
    holograms = h["holograms"][()]
    mask_pixel = h["mask_pixel"][()].astype(bool)
    support = h["supportmask"][()]
    material = h["material_thickness"][()]
    modes = h["modes"][()].tolist()
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    state_labels = h["state_labels"].asstr()[()].tolist()
    data_mode = h.attrs["data_mode"]
    xmcd_pairs = h["xmcd_pairs"][()]
    difference_reference_index = int(h.attrs["reference_index"])
    saved_focus_um = float(h.attrs.get("focus_prop_um", 0.))
    saved_focus_phase = float(h.attrs.get("focus_phase_rad", 0.))
    saved_focus_setup = None
    if "focus_pixel_size_m" in h.attrs:
        saved_focus_setup = {"px_size": h.attrs["focus_pixel_size_m"], "ccd_dist": h.attrs["focus_detector_distance_m"]}
    assert h.attrs["crop"] == 0 and h.attrs["binning"] == 1
# Resolve optional focus overrides before fitting; save the resolved values with the result.
if FOCUS_PROP_UM is not None:
    saved_focus_um = float(FOCUS_PROP_UM)
if FOCUS_PHASE_RAD is not None:
    saved_focus_phase = float(FOCUS_PHASE_RAD)
if DETECTOR_PIXEL_SIZE_M is not None or DETECTOR_DISTANCE_M is not None:
    saved_focus_setup = {} if saved_focus_setup is None else dict(saved_focus_setup)
    if DETECTOR_PIXEL_SIZE_M is not None:
        saved_focus_setup["px_size"] = float(DETECTOR_PIXEL_SIZE_M)
    if DETECTOR_DISTANCE_M is not None:
        saved_focus_setup["ccd_dist"] = float(DETECTOR_DISTANCE_M)
    if not all(key in saved_focus_setup for key in ("px_size", "ccd_dist")):
        raise ValueError("Provide both detector pixel size and distance, or inherit complete saved geometry")
if saved_focus_um != 0 and saved_focus_setup is None:
    raise ValueError("Nonzero focus requires detector pixel size and distance")
print("Physical focus (um, rad):", saved_focus_um, saved_focus_phase)
assert len(SATURATED_SCAN_INDICES) == len(SATURATION_SIGNS)
assert len(set(SATURATED_SCAN_INDICES)) == len(SATURATED_SCAN_INDICES)
assert set(SATURATED_SCAN_INDICES).issubset(set(scan_indices))
assert all(sign in (-1, 1) for sign in SATURATION_SIGNS)
saturated_states = {}
for index, sign in zip(SATURATED_SCAN_INDICES, SATURATION_SIGNS):
    state = state_labels[list(scan_indices).index(index)]
    if state in saturated_states and saturated_states[state] != sign:
        raise ValueError("The two polarizations of one state cannot have opposite saturation anchors")
    saturated_states[state] = float(sign)
calibrated = set(SATURATION_SIGNS) == {-1, 1}
print("Calibration:", "two saturated endpoints" if calibrated else "relative contrast (no two opposite anchors)")
reference_index = list(scan_indices).index(SATURATED_SCAN_INDICES[0]) if SATURATED_SCAN_INDICES else 0

recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    crop=CROP,
    binning=BINNING,
    roi=ROI,
    hologram_intensity_cutoff_vmin=HOLOGRAM_INTENSITY_CUTOFF_VMIN,
    partial_coherence=PARTIAL_COHERENCE,
    constrain_nonphysical_modes_common=COMMON_SECONDARY_MODES,
    warmup_Nit=WARMUP_NIT,
    projection_model=PROJECTION_MODEL,
    outer_iterations=OUTER_ITERATIONS,
    inner_mode=INNER_MODE,
    inner_Nit=INNER_ITERATIONS,
    beta_mode=BETA_MODE,
    beta_zero=BETA_ZERO,
    alpha_zero=ALPHA_ZERO,
    TV_freq=TV_FREQ,
    average_img=AVERAGE_IMG,
    physical_iterations=PHYSICAL_ITERATIONS,
    projection_relaxation=PROJECTION_RELAXATION,
    final_projection_relaxation=FINAL_PROJECTION_RELAXATION,
    final_fourier_constraint=FINAL_FOURIER_CONSTRAINT,
    fit_material_thickness=FIT_MATERIAL_THICKNESS,
    physical_projection_object_roi=PHYSICAL_PROJECTION_OBJECT_ROI,
    physical_phase_reference=PHYSICAL_PHASE_REFERENCE,
    initial_binary_magnetization=INITIAL_BINARY_MAGNETIZATION,
    binary_magnetization=BINARY_MAGNETIZATION,
    clip_magnetization=CLIP_MAGNETIZATION,
    magnetization_trend=MAGNETIZATION_TREND,
    magnetization_trend_intensity_cutoff=MAGNETIZATION_TREND_INTENSITY_CUTOFF,
    magnetization_trend_correction=MAGNETIZATION_TREND_CORRECTION,
    bound_magnetization_median=BOUND_MAGNETIZATION_MEDIAN,
    align_global_phase=ALIGN_GLOBAL_PHASE,
    fit_illumination_intensity=FIT_ILLUMINATION_INTENSITY,
    fit_aperture_illumination=FIT_APERTURE_ILLUMINATION,
    object_illumination_mode=OBJECT_ILLUMINATION_MODE,
    aperture_factor_type=APERTURE_FACTOR_TYPE,
    fft_workers=FFT_WORKERS,
    projection_focus_prop_um=saved_focus_um,
    projection_focus_phase_rad=saved_focus_phase,
    projection_focus_setup=saved_focus_setup,
    modes=modes,
    material_thickness=material,
    saturated_states=saturated_states,
    illumination_intensity_reference_observation=reference_index,
    projection_every=len(state_labels),
)
fields, received_start, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, support, state_labels=state_labels,
    energy_labels=energies, polarization_coefficients=polarizations,
    illumination_labels=["beam0"]*len(state_labels), universal_recipe=recipe,
    start_fields=warmup.copy(), phase_retrieval_kernel=universal.multimode_phase_retrieval_kernel,
)
np.testing.assert_allclose(received_start, warmup, rtol=1e-14, atol=0.)
assert list(components["state_names"]) == list(dict.fromkeys(state_labels))
print("Physical model log-fit RMS:", components["fit_residual_rms"])

In [ ]:
# Compute centered objects and measured-pixel residuals for saving.
mode_fields = fields[:, None] if fields.ndim == 3 else fields
objects = np.stack([universal.fourier_field_to_display_object(mode_fields[:, i])
                    for i in range(mode_fields.shape[1])], axis=1)
predicted_intensity = np.sum(abs(mode_fields)**2, axis=1)
valid = ~np.asarray(bsmasks, bool) & np.isfinite(holograms)
intensity_residual = np.sqrt(
    np.sum(np.where(valid, (predicted_intensity-holograms)**2, 0), axis=(1, 2))
    / np.maximum(np.sum(np.where(valid, holograms**2, 0), axis=(1, 2)), 1e-30))
# Center the fitted magnetization for the saved display maps.
magnetization = np.fft.fftshift(components["magnetization"], axes=(-2, -1))
active = material > 0
mean_magnetization = magnetization[:, active].mean(axis=1)
image_indices = np.arange(len(state_labels))  # Zero-based indices in acquisition order.
magnetic_state_indices = [list(components["state_names"]).index(label) for label in state_labels]
mean_magnetization_by_image = mean_magnetization[magnetic_state_indices]  # Both helicities share one state magnetization.
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["data_mode"] = data_mode
    h.attrs["reference_index"] = difference_reference_index
    h.create_dataset("xmcd_pairs", data=np.asarray(xmcd_pairs, dtype=int).reshape(-1, 2))
    h.create_dataset("magnetization_state_labels", data=components["state_names"], dtype=h5py.string_dtype())
    h.attrs["focus_prop_um"] = saved_focus_um
    h.attrs["focus_phase_rad"] = saved_focus_phase
    if saved_focus_setup is not None:
        h.attrs["focus_pixel_size_m"] = saved_focus_setup["px_size"]
        h.attrs["focus_detector_distance_m"] = saved_focus_setup["ccd_dist"]
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["calibration"] = "two_saturated_endpoints" if calibrated else "relative_contrast"
    h.attrs["component_coordinates"] = "components: solver fitting frame; magnetization: centered display frame"
    for name, array in [("fields", fields), ("warmup_fields", warmup), ("objects", objects),
                        ("holograms", holograms), ("mask_pixel", mask_pixel), ("bsmasks", bsmasks),
                        ("supportmask", support), ("material_thickness", material), ("modes", modes),
                        ("scan_indices", scan_indices), ("energy_eV", energies), ("polarization", polarizations),
                        ("magnetization", magnetization), ("mean_magnetization", mean_magnetization),
                        ("image_indices", image_indices), ("mean_magnetization_by_image", mean_magnetization_by_image),
                        ("intensity_residual", intensity_residual), ("saturated_scan_indices", SATURATED_SCAN_INDICES),
                        ("saturation_signs", SATURATION_SIGNS)]:
        h.create_dataset(name, data=array, compression="gzip")
    h.create_dataset("state_labels", data=state_labels, dtype=h5py.string_dtype())
    workflow.save_tree(h.create_group("components"), components)
    workflow.save_tree(h.create_group("errors"), errors)
    # Recipe settings and material are saved for reproducibility, not used as a UI.
    workflow.save_tree(h.create_group("recipe"), recipe)
print("Saved", OUTPUT)

## Plot XMCD or reference differences

XMCD: subtract negative polarization from positive polarization for each state. Single polarization: subtract the observation selected by `REFERENCE_INDEX` from every observation. Plot magnitude and real part of the complex difference.


In [ ]:
# Reload results; differences are formed from complex reconstructed objects.
RESULT_FILE = OUTPUT  # HDF5 result to inspect.
REFERENCE_INDEX = 0  # Plotting only: single-polarization reference observation.
DISPLAY_ROI = None  # Optional (row_start, row_stop, column_start, column_stop), object pixels.
ALIGN_DIFFERENCE_PHASE = True  # Match global phases in material-free reference holes before subtraction.
with h5py.File(RESULT_FILE, "r") as h:
    saved_fields = h["fields"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    labels = h["state_labels"].asstr()[()].tolist()
    support = h["supportmask"][()].astype(bool)
    material = h["material_thickness"][()]
    data_mode = h.attrs["data_mode"]
    reference_index = REFERENCE_INDEX
    xmcd_pairs = h["xmcd_pairs"][()]
    residuals = h["intensity_residual"][()]
    propagation_um = float(h.attrs.get("focus_prop_um", 0.))
    phase_rad = float(h.attrs.get("focus_phase_rad", 0.))
    focus_setup = None
    if "focus_pixel_size_m" in h.attrs:
        focus_setup = {"px_size": h.attrs["focus_pixel_size_m"], "ccd_dist": h.attrs["focus_detector_distance_m"]}
pairs = workflow.difference_pairs(data_mode, xmcd_pairs, reference_index, polarizations, energies)
reference_mask = support & (material == 0) if ALIGN_DIFFERENCE_PHASE else None
saved_objects = workflow.focused_objects(saved_fields, energies, propagation_um, phase_rad, focus_setup)
differences = workflow.object_differences(saved_objects, pairs, reference_mask)
view = np.s_[:, :] if DISPLAY_ROI is None else np.s_[DISPLAY_ROI[0]:DISPLAY_ROI[1], DISPLAY_ROI[2]:DISPLAY_ROI[3]]
difference_labels = [f"State {i}: + minus -" if data_mode == "xmcd" else f"{labels[a]} minus reference {b}" for i, (a, b) in enumerate(pairs)]
for mode in range(differences.shape[1]):
    selected = differences[:, mode, view[0], view[1]]
    limit = max(float(np.percentile(abs(selected), 99.5)), 1e-12)
    fig, axes = plt.subplots(len(pairs), 2, figsize=(10, 3.5*len(pairs)), squeeze=False)
    for i, title in enumerate(difference_labels):
        absolute = axes[i, 0].imshow(abs(selected[i]), vmin=0, vmax=limit)
        real = axes[i, 1].imshow(selected[i].real, cmap="RdBu_r", vmin=-limit, vmax=limit)
        axes[i, 0].set_title(f"{title}: absolute difference")
        axes[i, 1].set_title(f"{title}: real difference")
        fig.colorbar(absolute, ax=axes[i, 0], shrink=.75); fig.colorbar(real, ax=axes[i, 1], shrink=.75)
    fig.suptitle(f"Physical differences, mode {mode+1}"); fig.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(residuals, "o-"); ax.set(xlabel="Observation", ylabel="Relative intensity RMS", title="Measured-pixel agreement")
plt.show()


## Physical maps and comparison with warmup


In [ ]:
# Physical maps and illumination, read from the saved result.
with h5py.File(RESULT_FILE, "r") as h:
    magnetic_labels = h["magnetization_state_labels"].asstr()[()].tolist()
    magnetization = h["magnetization"][()]  # Already centered; do not shift it again.
    material = h["material_thickness"][()]
    mean_magnetization = h["mean_magnetization"][()]
    image_indices = h["image_indices"][()]
    mean_magnetization_by_image = h["mean_magnetization_by_image"][()]
    calibrated = h.attrs["calibration"] == "two_saturated_endpoints"
    common_wave = np.fft.fftshift(h["components/common_exit_waves"][()], axes=(-2, -1))
    illumination = h["components/illumination_intensity_factors"][()] if "illumination_intensity_factors" in h["components"] else np.ones(len(labels))
    warmup_fields = h["warmup_fields"][()]
active = material > 0
limit = max(1., float(np.max(abs(magnetization[:, active]))))
workflow.plot_stack(np.where(active, magnetization, np.nan)[:, view[0], view[1]], magnetic_labels,
                    "Reduced magnetization" if calibrated else "Relative magnetic contrast", "RdBu_r", -limit, limit)
workflow.plot_stack(abs(common_wave)[:, view[0], view[1]], ["beam0"], "Shared illumination/reference amplitude")
workflow.plot_stack(np.angle(common_wave)[:, view[0], view[1]], ["beam0"], "Shared illumination/reference phase", "twilight", -np.pi, np.pi)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(image_indices, mean_magnetization_by_image, "o-")
axes[0].set(xlabel="Image index (zero-based)", ylabel="Average magnetization" if calibrated else "Average relative magnetic contrast", title="Average over the material region")
axes[0].grid(alpha=.3)
# In XMCD mode the two polarization images have the same fitted state value.
axes[1].plot(illumination, "o-"); axes[1].set(xlabel="Observation", ylabel="Relative illumination intensity")
plt.show()

# Direct warmup/physical comparison for one selected observation.
COMPARE_INDEX = 0  # Zero-based observation for the before/after comparison.
assert 0 <= COMPARE_INDEX < len(labels)
warmup_modes = warmup_fields[:, None] if warmup_fields.ndim == 3 else warmup_fields
for mode in range(saved_objects.shape[1]):
    before = workflow.focused_objects(warmup_modes[COMPARE_INDEX:COMPARE_INDEX+1], energies[COMPARE_INDEX:COMPARE_INDEX+1], propagation_um, phase_rad, focus_setup)[0, mode][view]
    after = saved_objects[COMPARE_INDEX, mode][view]
    amplitude_limit = max(float(np.percentile(np.stack([abs(before), abs(after)]), 99.5)), 1e-12)
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for row, obj, title in zip(range(2), [before, after], ["Warmup", "Physical"]):
        amplitude = axes[row, 0].imshow(abs(obj), vmin=0, vmax=amplitude_limit)
        phase = axes[row, 1].imshow(np.angle(obj), cmap="twilight", vmin=-np.pi, vmax=np.pi)
        axes[row, 0].set_title(f"{title} amplitude"); axes[row, 1].set_title(f"{title} phase")
        fig.colorbar(amplitude, ax=axes[row, 0], shrink=.75); fig.colorbar(phase, ax=axes[row, 1], shrink=.75)
    fig.suptitle(f"{labels[COMPARE_INDEX]}, mode {mode+1}: warmup versus physical")
    fig.tight_layout(); plt.show()
